# 拓展：RFM用户分群——从"切8份"到"可验证的自动化分群系统"

> **本拓展新增 7 个课程未覆盖的深度概念**
> 关联主课：W2D5 周五_RFM用户分群
> 沙盘：鲜达生鲜（生鲜电商平台）
> 你的角色：数据运营负责人
> 建议时间：2-2.5 小时

---

## 知识网络预览

```
主课核心: RFM三维 -> 中位数切分 -> 2x2x2=8群体 -> 运营策略
         |
    +----+----+----------+----------+----------+
    |    |    |          |          |          |
  [前置] [并行] [防护]     [防护]     [落地]     [替代]+[落地]
  K-Means 轮廓  边界敏感性  数据质量   RFM完整   健康度评分
  自动分箱  系数  翻转率量化  三重污染   SQL实现   +产品化SPC
    |      |      |        |        |        |
    +------+------+--------+--------+--------+
                      |
         全通向：让RFM从"手动切8份"升级为"可验证、可监控、可自动化的分群系统"
```

**阅读指引**：
- 🟢 [必修] 概念1-5 — RFM从手工到自动化的核心链路
- 🟡 [选修] 概念6-7 — RFM的边界与工程化补充
- 入口自诊（第二部分）先做，它会告诉你哪些概念你最需要仔细读


---

# 第一部分：沙盘设定

## 公司画像

**鲜达**，成立4年的生鲜电商平台。覆盖全国200+城市，300万注册用户，月活80万。

商业模式：自营仓配(60%GMV，次日达) + 平台模式(40%GMV，本地商户入驻，抽佣8-15%)。核心品类：水果蔬菜(35%)、肉禽蛋奶(25%)、海鲜水产(15%)、粮油调味(15%)、其他(10%)。

用户以25-45岁城市家庭为主，女性占65%。客单价约85元，月均购买2.3次。


## 当前挑战

运营总监赵总在月度复盘会上拍了桌子：

> "RFM已经跑了三个月了，但最近出了三个问题让我睡不着觉！"

你调出数据后，发现三个隐藏炸弹：

1. **20%的"重要价值"用户标签在两个月内发生了变化** — 不是因为用户行为变了，而是因为他们刚好骑在中位数边界上。产品经理钱经理投诉："上周还是VIP，这周就变成了'一般价值'——这让我怎么跟用户解释？"

2. **双11期间M值被大幅污染** — 大促囤货用户(一次买半年量的洗衣液)被标记为"重要价值"，但接下来5个月都不会再买。运营给这批人发了VIP权益——纯浪费。

3. **每周手动跑一次RFM脚本** — 数据工程师孙工周五下午什么事都不干就导出Excel->Python->Excel->发邮件。用户量快破300万了，这个流程迟早崩。

赵总要你在两周内给出：
- 分箱方法能不能比"中位数一刀切"更好？
- 怎么自动检测被污染的数据？
- RFM能不能自动化到"每天早上运营打开看板就知道该给谁发券"？


## 数据资产

你手上有以下数据表（后续代码中将用pandas模拟）：

| 表名 | 关键字段 | 说明 |
|------|---------|------|
| `orders` | order_id, user_id, order_date, amount, category, is_returned | 订单主表(日均5万单) |
| `order_items` | order_id, product_id, quantity, unit_price, is_promotion | 订单明细 |
| `users` | user_id, register_date, channel, city, member_level | 用户基本信息 |
| `user_behavior` | user_id, event_date, event_type, event_count | 用户行为日志(浏览/加购/搜索) |
| `returns` | return_id, order_id, user_id, return_date, return_amount | 退货记录 |
| `coupons` | coupon_id, user_id, coupon_amount, sent_date, used_date | 优惠券发放与核销 |
| `rfm_labels` | user_id, r_value, f_value, m_value, rfm_label, update_date | 现有RFM标签表 |
| `promotions` | promo_id, start_date, end_date, type, discount_pct | 大促活动日历 |

## 利益相关方

| 角色 | 关心什么 | 可能的冲突 |
|------|---------|----------|
| 运营总监 赵总 | RFM分群不要再"跳来跳去" | "边界用户占了快15%，他们的标签每个月都在变" |
| 产品经理 钱经理 | 能不能给每个用户一个稳定的"画像" | "中位数太粗暴了——能不能用机器学习分群？" |
| 数据工程师 孙工 | 自动化！自动化！自动化！ | "每周五下午都在跑RFM，我想做点更有价值的事" |
| 市场VP 周总 | 大促后的RFM数据还准不准？ | "双11后'重要价值'用户翻了一倍——这不正常吧？" 


## 你的角色

你是鲜达的**数据运营负责人**。你的任务不是"再跑一次RFM"——你已经在跑了。你的任务是**把RFM从一个"周报工具"升级为一个"可验证、可监控、可自动化"的分群系统**：

1. **诊断**：中位数分箱到底有什么问题？边界用户占比多少？标签稳定性如何？
2. **升级**：K-Means聚类能不能替代中位数分箱？怎么评估聚类质量？
3. **防污染**：怎么自动检测退货/拆单/大促对RFM的污染？
4. **工程化**：RFM从Python脚本变成SQL定时任务+监控看板需要什么？


---

# 第二部分：入口自诊（5分钟）

以下5题测试你对**主课内容**的掌握程度。做完后根据结果，你会知道自己最需要在哪个拓展概念上花时间。


### 自诊题

**Q1**：RFM三个维度中，哪个维度在预测用户"下次会不会买"时通常预测力最强？为什么？

<details><summary>点击查看答案</summary>

**参考答案**：R（Recency）——最近一次消费距今多久。因为消费行为有强惯性——昨天刚买的人明天更可能再买。主课1.3节给出了实证数据：R的预测力(AUC约0.75)远高于F(AUC约0.60)和M(AUC约0.55)。

**常见偏差**：
1. 选M——"花钱多的人更忠诚"是一种直觉但数据不支持的假设
2. 选F——高频但最近没来（R高）的用户可能已经找到了替代品

**补救路径**：回主课 1.3 节"从第一性原理推导"。
</details>

**Q2**：为什么RFM用乘法交叉(2x2x2=8)而不是加法评分(如R+F+M各打1-5分，总分3-15)？

<details><summary>点击查看答案</summary>

**参考答案**：加法评分会产生"补偿效应"——高M可以补偿低R，导致"很久没来但花过大钱"的用户和"最近常来且花大钱"的用户得到相同总分。乘法交叉保证每个维度独立判断，不互相补偿。

**常见偏差**：
1. "加法评分更精细(15个等级vs8个群体)"——但15个等级中有很多"多对一映射"，不同用户画像被合并了
2. "打分制更科学"——看似科学，实则掩盖了每个维度的独立意义

**补救路径**：回主课 1.6 节"RFM的数学本质"。
</details>

**Q3**：用户A的R=18天, F=5次, M=600元。中位数分别为R=20天, F=3次, M=52元。用户A的RFM标签是什么？

<details><summary>点击查看答案</summary>

**参考答案**：重要价值。R=18天 <= 中位20天 -> R_high=1；F=5次 >= 中位3次 -> F_high=1；M=600元 >= 中位52元 -> M_high=1。三个都是1 -> 重要价值。

**常见偏差**：R方向搞反——R=18天比20天小(更活跃)，面试中R方向搞反基本就挂了。

**补救路径**：回主课 2.1 节"高/低分界怎么定"。
</details>

**Q4**：RFM迁移矩阵中，"重要价值->重要挽留"的迁移意味着什么？应该怎么应对？

<details><summary>点击查看答案</summary>

**参考答案**：F和M仍然高，但R变差了——高价值用户最近不来了。这是最高优先级流失预警。应对：24小时内发个性化push，第3天短信，第7天电话，用大额券挽回。

**常见偏差**："发张券就完了"——对高价值用户，群发的效果远不如个性化沟通。

**补救路径**：回主课 2.8 节"迁移矩阵"。
</details>

**Q5**：某内容社区的RFM改写中，M应该改成什么？为什么不能直接用消费金额？

<details><summary>点击查看答案</summary>

**参考答案**：M改为"近30天互动量(获赞+评论+收藏)"或"近30天内容消费量"。因为创作者的核心价值不是消费金额而是内容产出。"一个月发30篇爆款但从不买东西"的创作者是社区最宝贵的资产。

**常见偏差**：M改成"使用时长"——时长是过程指标，不代表产出；M改成"关注数"——是存量而非近期增量。

**补救路径**：回主课 2.11 节"非交易型产品的RFM改写"。
</details>

---

**自诊结果解读**：
- 全部答对(5/5) -> 主课基础扎实，直接从第三部分开始
- 3-4题对 -> 先快速复习主课 1.3/1.6/2.1/2.8/2.11 五个小节（10min）
- 0-2题对 -> 建议先回主课完整过一遍再来看拓展


---

# 第三部分：概念网络展开（50-70分钟）

## 概念总览

| # | 概念 | 关系类型 | 重要度 | 深度类型 | 解决鲜达的什么问题 |
|---|------|---------|--------|---------|-----------------|
| 1 | K-Means自动化分箱 vs 中位数 | [前置] | 必修 | [M][C] | 钱经理的"中位数太粗暴" |
| 2 | 轮廓系数+肘部法则 | [并行] | 必修 | [M][C] | "到底分几群最好？" |
| 3 | RFM标签边界敏感性：翻转率量化 | [防护] | 必修 | [M][C] | 赵总的"标签跳来跳去" |
| 4 | RFM数据质量三重污染 | [防护] | 必修 | [M][C] | 周总的"大促后数据不准" |
| 5 | RFM完整SQL生产实现 | [落地] | 必修 | [C] | 孙工的"自动化！" |
| 6 | 客户健康度评分 vs RFM | [替代] | 选修 | [B][C] | "只看交易够不够？" |
| 7 | RFM产品化：指标身份证+SPC监控 | [落地] | 选修 | [M][C] | "从周报到每日监控看板" |

**概念总数**：7个（[必修]5 [选修]2）
**深度配比**：代码+数学深度 = 5/7 = 71% >= 60%
**空白填补**：3个概念来自全课程永久空白区


---

### 概念1：K-Means自动化分箱 vs 中位数一刀切 [前置] [必修]

#### 认知冲突

鲜达的钱经理找到你："上个月有个用户R=21天，F=2次，M=55元——被标为'一般挽留'。这个月ta的R=19天，F=2次，M=54元——变成了'新用户/低价值'。ta只是少花了1块钱、早来了2天，标签就全变了。能不能不要这么粗暴？"

你打开数据一看——中位数R=20天，F=3次，M=52元。**20%的用户在至少一个维度上距离中位数不超过10%。** 他们这个月在"高"，下个月在"低"——不是因为行为变了，而是因为**中位数本身就是一刀切**。

**主课教了"用中位数切"，但没教"中位数切有什么问题"以及"有没有更好的切法"。**


#### 概念引入

**一句话白话**：K-Means聚类就是"让数据自己告诉你应该怎么分组"——相似的R/F/M值归到同一个群，而不是人为规定"高于中位数=高"。

**核心逻辑 + 具体例子**

中位数一刀切的问题：
```
用户A: R=19天, F=3次, M=53元 -> R_high=1(<=20), F_high=1(>=3), M_high=1(>=52) -> 重要价值
用户B: R=21天, F=3次, M=53元 -> R_high=0(>20), F_high=1(>=3), M_high=1(>=52) -> 重要挽留
```
A和B的R值只差2天，但标签完全不同。中位数把连续的行为强行二值化了。

K-Means的思路：
1. 随机选k个点作为"群中心"
2. 每个用户分配给最近的群中心
3. 重新计算每个群的"中心"
4. 重复步骤2-3直到中心不再移动

在鲜达的场景中，如果我们用K-Means(k=8)替代中位数分箱：
- 群中心是从数据中学出来的——不需要人为规定"R<=20天=高"
- 用户A和B大概率在同一个群（因为他们的R/F/M值非常接近）
- 标签更稳定——因为群边界是基于数据密度而非硬阈值

**与核心概念的关系**：[前置] — 主课教了"怎么用中位数切8份"，本概念教"在切之前——中位数是不是最好的切法？K-Means能不能替代它？"。这是从"执行RFM"到"设计RFM分箱方法"的升级。

> **不是替代，是补充**：中位数法的最大优势是可解释性（运营人员秒懂）。K-Means更精确但像"黑箱"。实际工作中，建议**用K-Means探索最优分群结构，用分位数做业务落地**——两者配合使用。


#### 在沙盘中的用武之地

鲜达的运营团队每周按RFM标签发券。如果20%的用户因为中位数边界问题被错误分群->每100万用户中有20万人的运营策略可能是错的。K-Means自动发现数据中的自然聚类->边界更合理->标签更稳定。

更重要的是：K-Means可以帮我们发现"8群是不是最优的"——也许鲜达的用户实际上是5群或10群更合理？这是中位数法无法回答的问题。


In [ ]:
# ============================================================
# 概念1 练习：K-Means自动化分箱 vs 中位数一刀切
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---- 生成鲜达的模拟RFM数据 ----
np.random.seed(42)
n_users = 2000

# 生鲜用户的行为模式：5个自然群体
true_clusters = np.random.choice([0,1,2,3,4], n_users, p=[0.10, 0.30, 0.25, 0.20, 0.15])

cluster_params = {
    0: {'R_mean': 3,  'R_std': 2,  'F_mean': 12, 'F_std': 3,  'M_mean': 800,  'M_std': 200},
    1: {'R_mean': 10, 'R_std': 5,  'F_mean': 6,  'F_std': 2,  'M_mean': 300,  'M_std': 100},
    2: {'R_mean': 25, 'R_std': 10, 'F_mean': 2,  'F_std': 1,  'M_mean': 80,   'M_std': 40},
    3: {'R_mean': 2,  'R_std': 1,  'F_mean': 3,  'F_std': 1,  'M_mean': 200,  'M_std': 80},
    4: {'R_mean': 50, 'R_std': 20, 'F_mean': 1,  'F_std': 1,  'M_mean': 40,   'M_std': 30},
}

R_vals = np.zeros(n_users); F_vals = np.zeros(n_users); M_vals = np.zeros(n_users)
for i in range(n_users):
    c = true_clusters[i]; p = cluster_params[c]
    R_vals[i] = max(0, np.random.normal(p['R_mean'], p['R_std']))
    F_vals[i] = max(0, np.random.normal(p['F_mean'], p['F_std']))
    M_vals[i] = max(1, np.random.normal(p['M_mean'], p['M_std']))

df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'R': R_vals.astype(int), 'F': F_vals.astype(int), 'M': M_vals.astype(int),
    'true_cluster': true_clusters
})

# ---- 方法1: 中位数一刀切 ----
rm = df['R'].median(); fm = df['F'].median(); mm = df['M'].median()
df['R_high'] = (df['R'] <= rm).astype(int)
df['F_high'] = (df['F'] >= fm).astype(int)
df['M_high'] = (df['M'] >= mm).astype(int)

def rfm_label(r, f, m):
    if r==1 and f==1 and m==1: return '重要价值'
    elif r==0 and f==1 and m==1: return '重要挽留'
    elif r==1 and f==0 and m==1: return '重要发展'
    elif r==0 and f==0 and m==1: return '流失高价值'
    elif r==1 and f==1 and m==0: return '一般价值'
    elif r==0 and f==1 and m==0: return '一般挽留'
    elif r==1 and f==0 and m==0: return '新用户/低价值'
    else: return '流失用户'

df['rfm_median'] = df.apply(lambda r: rfm_label(r['R_high'], r['F_high'], r['M_high']), axis=1)

print("=" * 65)
print("方法1: 中位数一刀切")
print("=" * 65)
print(f"分界标准: R<={rm:.0f}天, F>={fm:.0f}次, M>={mm:.0f}元")
print(f"\n群体分布:")
print(df['rfm_median'].value_counts())

# ---- 方法2: K-Means聚类 ----
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df[['R', 'F', 'M']])
rfm_scaled[:, 0] = -rfm_scaled[:, 0]  # R越小越好，取反

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(rfm_scaled)

# 给每个K-Means群命名
cluster_stats = df.groupby('kmeans_cluster').agg(
    avg_R=('R', 'mean'), avg_F=('F', 'mean'), avg_M=('M', 'mean'), size=('user_id', 'count')
).round(1)
cluster_stats['score'] = (-cluster_stats['avg_R'] / cluster_stats['avg_R'].std() +
                            cluster_stats['avg_F'] / cluster_stats['avg_F'].std() +
                            cluster_stats['avg_M'] / cluster_stats['avg_M'].std())
cluster_stats = cluster_stats.sort_values('score', ascending=False)

cluster_names = ['核心高价值', '稳定活跃', '高频低额', '近期活跃',
                 '中频中额', '低频低额', '流失边缘', '沉睡用户']
cluster_name_map = {old: new for old, new in zip(cluster_stats.index, cluster_names)}
df['kmeans_label'] = df['kmeans_cluster'].map(cluster_name_map)

print("\n" + "=" * 65)
print("方法2: K-Means聚类 (k=8)")
print("=" * 65)
for name in cluster_names:
    mask = df['kmeans_label'] == name
    if mask.sum() > 0:
        print(f"  {name}: R均={df.loc[mask,'R'].mean():.0f}天, "
              f"F均={df.loc[mask,'F'].mean():.1f}次, M均={df.loc[mask,'M'].mean():.0f}元, "
              f"人数={mask.sum()}")

# ---- 边界用户检测 ----
pct_margin = 0.10
boundary_r = (df['R'] > rm * (1 - pct_margin)) & (df['R'] < rm * (1 + pct_margin))
boundary_f = (df['F'] > fm * (1 - pct_margin)) & (df['F'] < fm * (1 + pct_margin))
boundary_m = (df['M'] > mm * (1 - pct_margin)) & (df['M'] < mm * (1 + pct_margin))
boundary_users = boundary_r | boundary_f | boundary_m

print(f"\n=== 边界用户分析 ===")
print(f"中位数边界+/-10%范围内的用户: {boundary_users.sum()}人 ({boundary_users.sum()/n_users:.1%})")
print(f"  - R边界: {boundary_r.sum()}人")
print(f"  - F边界: {boundary_f.sum()}人")
print(f"  - M边界: {boundary_m.sum()}人")

# ---- 可视化 ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
colors_map = {'重要价值': '#e74c3c', '重要挽留': '#e67e22', '重要发展': '#f1c40f',
              '流失高价值': '#9b59b6', '一般价值': '#2ecc71', '一般挽留': '#1abc9c',
              '新用户/低价值': '#3498db', '流失用户': '#95a5a6'}
for label in colors_map:
    mask = df['rfm_median'] == label
    ax.scatter(df.loc[mask, 'R'], df.loc[mask, 'F'], c=colors_map[label],
               label=label, alpha=0.5, s=10)
ax.axvline(x=rm, color='red', linestyle='--', linewidth=2, label=f'R中位={rm:.0f}')
ax.axhline(y=fm, color='orange', linestyle='--', linewidth=2, label=f'F中位={fm:.0f}')
ax.set_xlabel('R (最近购买距今/天)'); ax.set_ylabel('F (购买次数)')
ax.set_title('中位数法: R-F平面 (硬边界)'); ax.legend(fontsize=6, loc='upper right')

ax = axes[1]
colors_kmeans = plt.cm.tab10(np.linspace(0, 1, 8))
for i, name in enumerate(cluster_names):
    mask = df['kmeans_label'] == name
    ax.scatter(df.loc[mask, 'R'], df.loc[mask, 'F'], c=[colors_kmeans[i]],
               label=name, alpha=0.5, s=10)
ax.set_xlabel('R (最近购买距今/天)'); ax.set_ylabel('F (购买次数)')
ax.set_title('K-Means: R-F平面 (自然边界)'); ax.legend(fontsize=6, loc='upper right')

ax = axes[2]
ax.scatter(df.loc[~boundary_users, 'R'], df.loc[~boundary_users, 'F'],
           c='gray', alpha=0.2, s=8, label='非边界用户')
ax.scatter(df.loc[boundary_users, 'R'], df.loc[boundary_users, 'F'],
           c='red', alpha=0.6, s=15, label=f'边界用户({boundary_users.sum()}人)')
ax.axvline(x=rm, color='red', linestyle='--', alpha=0.5)
ax.axhline(y=fm, color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('R (最近购买距今/天)'); ax.set_ylabel('F (购买次数)')
ax.set_title(f'边界用户: 骑在中位数+/-10%范围内')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('kmeans_vs_median_rfm.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存: kmeans_vs_median_rfm.png")


#### 练习反馈

**参考答案**：
- 中位数法：约20-25%的用户在至少一个维度上处于中位数+/-10%的边界
- K-Means：群中心由数据密度决定，边界用户更少（因为没有人为硬阈值）
- 两个方法的大方向一致但边界用户归属不同

**常见偏差**：
1. "K-Means完全替代中位数法" — K-Means的群中心不好解释，运营人员无法像"R<=20天=高"那样直观理解。**实际工作中K-Means做探索，分位数做落地。**
2. "k=8就行了" — K-Means需要我们自己指定k。为什么是8？中位数法强制8份，但数据中可能只有5个自然群体。下一个概念解决"怎么选k"。
3. "标准化不重要" — R/F/M的量纲完全不同(R是天，M是元)，不标准化会导致M主导聚类结果。

**补救路径**：
- 如果聚类结果解释不了 -> 尝试先对每个维度单独聚类，再组合
- 如果想要运营可解释 -> 用分位数(如0-25%/25-50%/50-75%/75-100%)替代K-Means


---

### 概念2：聚类质量评估——轮廓系数 + 肘部法则 [并行] [必修]

#### 认知冲突

你在概念1中用K-Means(k=8)做了聚类，但钱经理追问了一句："为什么是8群？因为中位数法也是8群？那这不算'数据自己说话'——你在用中位数法的结论来决定K-Means的参数。"

你愣住了。对啊——如果你先入为主地设k=8，K-Means的优势就没了。**但k到底应该是几？** 3群？5群？10群？每种选择都有人支持，但没有数据依据。


#### 概念引入

**一句话白话**：轮廓系数(Silhouette Score)就是"群内的用户有多像，群间的用户有多不像"的量化打分——分数越高说明聚类越好。肘部法则(Elbow Method)是"k增加带来的边际收益递减"的可视化——找到"收益不再明显增加"的那个拐点。

**核心逻辑 + 具体例子**

**轮廓系数**（取值范围[-1, +1]）：
- +1 = 用户离同群其他人很近，离最近的其他群很远 -> 完美聚类
- 0 = 用户刚好在两个群的边界上 -> 归属模糊
- -1 = 用户被分到了错误的群

公式：`s = (b - a) / max(a, b)`
- a = 用户到同群其他用户的平均距离（越小越好——群内紧凑）
- b = 用户到最近的其他群的平均距离（越大越好——群间分离）

**肘部法则**：
- 计算k=2到k=15每个k值下的"簇内平方和(Inertia)"
- Inertia随着k增加必然下降(群越多每群越小->群内距离越小)
- 但下降速度会在"最优k"处突然变慢——形成"肘部"

在鲜达的300万用户中，我们分别对k=2到15做K-Means，找到轮廓系数最高的k和肘部拐点。如果两个指标都指向k=5——那鲜达的用户最好分5群而不是8群。

**与核心概念的关系**：[并行] — 概念1教了"怎么用K-Means"，本概念教"怎么判断K-Means做得好不好"。没有聚类评估，K-Means的k就是拍脑袋。


#### 在沙盘中的用武之地

鲜达的300万用户——如果最优k=5，运营商只需要管理5个群体(而不是8个)，每个群体更大(约60万人/群)，策略设计更聚焦。如果最优k=10，说明用户差异比预想的更细——需要更精细的运营策略。选错k的代价：k太小->不同用户混在一起；k太大->每个群太小(运营团队管不过来)。


In [ ]:
# ============================================================
# 概念2 练习：轮廓系数 + 肘部法则 选择最优k
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---- 复用概念1的数据准备 ----
np.random.seed(42)
n_users = 2000
true_clusters = np.random.choice([0,1,2,3,4], n_users, p=[0.10, 0.30, 0.25, 0.20, 0.15])
cluster_params = {
    0: {'R_mean': 3, 'R_std': 2, 'F_mean': 12, 'F_std': 3, 'M_mean': 800, 'M_std': 200},
    1: {'R_mean': 10, 'R_std': 5, 'F_mean': 6, 'F_std': 2, 'M_mean': 300, 'M_std': 100},
    2: {'R_mean': 25, 'R_std': 10, 'F_mean': 2, 'F_std': 1, 'M_mean': 80, 'M_std': 40},
    3: {'R_mean': 2, 'R_std': 1, 'F_mean': 3, 'F_std': 1, 'M_mean': 200, 'M_std': 80},
    4: {'R_mean': 50, 'R_std': 20, 'F_mean': 1, 'F_std': 1, 'M_mean': 40, 'M_std': 30},
}
R_vals = np.zeros(n_users); F_vals = np.zeros(n_users); M_vals = np.zeros(n_users)
for i in range(n_users):
    c = true_clusters[i]; p = cluster_params[c]
    R_vals[i] = max(0, np.random.normal(p['R_mean'], p['R_std']))
    F_vals[i] = max(0, np.random.normal(p['F_mean'], p['F_std']))
    M_vals[i] = max(1, np.random.normal(p['M_mean'], p['M_std']))

df_rfm = pd.DataFrame({'R': R_vals, 'F': F_vals, 'M': M_vals})
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df_rfm)
rfm_scaled[:, 0] = -rfm_scaled[:, 0]

# ---- 遍历k=2到15 ----
k_range = range(2, 16)
inertias = []; silhouette_scores_list = []

print("=" * 65)
print("K-Means聚类质量评估: k=2 到 k=15")
print("=" * 65)
print(f"{'k':<6} {'Inertia':<14} {'轮廓系数':<14} {'评价'}")
print("-" * 65)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(rfm_scaled, labels) if k > 1 else 0
    silhouette_scores_list.append(sil)
    comment = ''
    if k > 2 and sil >= max(silhouette_scores_list[:-1]):
        comment = '<-- 新高!'
    print(f"{k:<6} {km.inertia_:<14.0f} {sil:<14.4f} {comment}")

best_k_sil = list(k_range)[silhouette_scores_list.index(max(silhouette_scores_list))]
print(f"\n>>> 轮廓系数最高: k={best_k_sil} (分数={max(silhouette_scores_list):.4f})")

# ---- 肘部法则 ----
deltas = np.diff(inertias)
delta2 = np.diff(deltas)
elbow_k = list(k_range)[np.argmax(delta2) + 2]
print(f">>> 肘部法则拐点: k≈{elbow_k}")

# ---- 可视化 ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(list(k_range), inertias, 'bo-', linewidth=2)
ax.axvline(x=elbow_k, color='red', linestyle='--', label=f'肘部拐点 k≈{elbow_k}')
ax.set_xlabel('k (聚类数)'); ax.set_ylabel('簇内平方和 (Inertia)')
ax.set_title('肘部法则'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(list(k_range), silhouette_scores_list, 'go-', linewidth=2)
ax.axvline(x=best_k_sil, color='green', linestyle='--', label=f'最佳k={best_k_sil}')
ax.set_xlabel('k (聚类数)'); ax.set_ylabel('轮廓系数')
ax.set_title('轮廓系数'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
km_best = KMeans(n_clusters=best_k_sil, random_state=42, n_init=10)
labels_best = km_best.fit_predict(rfm_scaled)
sample_sil_values = silhouette_samples(rfm_scaled, labels_best)
y_lower = 10
sil_avg = silhouette_score(rfm_scaled, labels_best)
for i in range(best_k_sil):
    ith_cluster_sil = sample_sil_values[labels_best == i]
    ith_cluster_sil.sort()
    size_cluster_i = len(ith_cluster_sil)
    y_upper = y_lower + size_cluster_i
    color = plt.cm.tab10(float(i) / best_k_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil,
                      facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, f'群{i}')
    y_lower = y_upper + 10
ax.axvline(x=sil_avg, color='red', linestyle='--', label=f'平均轮廓={sil_avg:.3f}')
ax.set_xlabel('轮廓系数'); ax.set_ylabel('用户')
ax.set_title(f'k={best_k_sil}时各群的轮廓分布')
ax.legend(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('kmeans_quality_assessment.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存: kmeans_quality_assessment.png")

print("\n" + "=" * 65)
print("对鲜达的建议")
print("=" * 65)
print(f"1. 数据驱动的建议k = {best_k_sil} (轮廓系数法) 或 k≈{elbow_k} (肘部法则)")
if best_k_sil != 8:
    print(f"2. 重要发现: 最优k≠8! 当前中位数法的8群体可能不是最自然的用户分层")
    print(f"   建议: 用k={best_k_sil}做运营策略设计，每个群体更大更聚焦")
print(f"3. 不要只看数字: 如果k={best_k_sil}解释不通->退回到业务能理解的k")


#### 练习反馈

**参考答案**：
- 轮廓系数通常在k=4~6之间达到峰值（取决于随机种子，通常在k=5附近）
- 肘部拐点也在k≈4~5附近
- 最优k≠8——说明鲜达的用户自然分成约5群，8群是人为切出来的

**常见偏差**：
1. "只看轮廓系数" — 轮廓系数在大k时可能仍然高。需要结合肘部法则：肘部拐点后的k增加收益递减。
2. "k=轮廓系数最高的那个，其他不考虑" — 如果轮廓系数最高的k=15，运营团队不可能管理15个群体。统计最优 != 业务最优。
3. "k=5说明8群法是错的" — 不是对错问题。中位数法适合需要固定群体数量的运营SOP；K-Means适合探索真实用户结构。

**补救路径**：
- 如果两个指标推荐的k不同 -> 取中间值，分别测试->选运营团队能理解的
- 如果轮廓系数<0.2 -> 数据中可能没有明显聚类结构->不适合K-Means->回到分位数法


---

### 概念3：RFM标签边界敏感性——翻转率量化 [防护] [必修]

#### 认知冲突

赵总把两个月的RFM报告扔到你桌上："上个月'重要价值'有161人，这个月只有131人——少了30人。这30人是'流失了'还是'标签变了'？如果是标签变了——他们真的变了吗，还是仅仅因为中位数移动了？"

你翻出原始数据：两个月的标签相同的只有617人(61.7%)。38.3%的用户标签发生了变化。但他们的行为真的变了38.3%吗？还是因为**中位数在移动 + 边界用户在翻转**？


#### 概念引入

**一句话白话**：边界敏感性量化的是"用户稍微动一动(或中位数稍微变一变)，标签就变了"的概率。这个概率越高，RFM系统越不可信。

**核心逻辑 + 具体例子**

RFM标签翻转有三个来源：

**来源A：用户行为真的变了**（这是RFM应该捕捉的，是"有效翻转"）
- 用户从每周买->3个月没买 -> 标签从"重要价值"->"重要挽留" -> 正确的信号

**来源B：中位数移动了**（这不是用户变了，是"尺子"变了）
- 用户A的R=21天。上月中位=20天->R_high=0；本月中位=22天->R_high=1
- 用户A什么都没变，但因为中位数移动，标签变了

**来源C：用户刚好骑在边界上**（最隐蔽的问题）
- 用户C的R=21天。上月中位=20天->R_high=0；ta多来了一次->R=18天->R_high=1
- 行为变化只有3天，但标签全变了

**翻转率 = 标签改变的用户数 / 总用户数**。我们需要把它拆成"有效翻转"(来源A)和"噪声翻转"(来源B+C)。如果噪声翻转占比>50%，RFM系统的可靠性就值得怀疑。

**与核心概念的关系**：[防护] — 主课教了"怎么贴标签"，本概念教"标签贴完之后——这个标签有多可靠？"。标签稳定性是RFM系统信任度的基础。

**自包含说明**：翻转率计算只需要crosstab(迁移矩阵)+判断每个用户三个维度的变化量是否超过阈值。不需要额外的统计知识。


#### 在沙盘中的用武之地

鲜达的赵总质疑"标签跳来跳去"——你需要量化这个问题的严重程度：
1. 总翻转率是多少？
2. 其中多少是"有效翻转"(用户行为真的跨过了合理的阈值)？
3. 多少是"噪声翻转"(中位数移动+边界抖动)？
4. 针对噪声翻转——能不能通过"锁定上月分界标准"来消除来源B？


In [ ]:
# ============================================================
# 概念3 练习：RFM标签边界敏感性分析
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(99)
n_users = 2000

# 模拟两期RFM数据
true_clusters = np.random.choice([0,1,2,3,4], n_users, p=[0.10, 0.30, 0.25, 0.20, 0.15])
cluster_params = {
    0: {'R_mean': 3, 'R_std': 2, 'F_mean': 12, 'F_std': 3, 'M_mean': 800, 'M_std': 200},
    1: {'R_mean': 10, 'R_std': 5, 'F_mean': 6, 'F_std': 2, 'M_mean': 300, 'M_std': 100},
    2: {'R_mean': 25, 'R_std': 10, 'F_mean': 2, 'F_std': 1, 'M_mean': 80, 'M_std': 40},
    3: {'R_mean': 2, 'R_std': 1, 'F_mean': 3, 'F_std': 1, 'M_mean': 200, 'M_std': 80},
    4: {'R_mean': 50, 'R_std': 20, 'F_mean': 1, 'F_std': 1, 'M_mean': 40, 'M_std': 30},
}

def generate_rfm(seed_offset=0):
    np.random.seed(99 + seed_offset)
    R = np.zeros(n_users); F = np.zeros(n_users); M = np.zeros(n_users)
    for i in range(n_users):
        c = true_clusters[i]; p = cluster_params[c]
        R[i] = max(0, np.random.normal(p['R_mean'] + seed_offset, p['R_std']))
        F[i] = max(0, np.random.normal(p['F_mean'], p['F_std']))
        M[i] = max(1, np.random.normal(p['M_mean'], p['M_std']))
    return R.astype(int), F.astype(int), M.astype(int)

def rfm_label(R, F, M, r_med, f_med, m_med):
    r_high = (R <= r_med).astype(int)
    f_high = (F >= f_med).astype(int)
    m_high = (M >= m_med).astype(int)
    labels = []
    for rh, fh, mh in zip(r_high, f_high, m_high):
        if rh==1 and fh==1 and mh==1: labels.append('重要价值')
        elif rh==0 and fh==1 and mh==1: labels.append('重要挽留')
        elif rh==1 and fh==0 and mh==1: labels.append('重要发展')
        elif rh==0 and fh==0 and mh==1: labels.append('流失高价值')
        elif rh==1 and fh==1 and mh==0: labels.append('一般价值')
        elif rh==0 and fh==1 and mh==0: labels.append('一般挽留')
        elif rh==1 and fh==0 and mh==0: labels.append('新用户/低价值')
        else: labels.append('流失用户')
    return np.array(labels)

R_prev, F_prev, M_prev = generate_rfm(seed_offset=5)
R_curr, F_curr, M_curr = generate_rfm(seed_offset=0)

# 方法A: 每月用当月中位数
label_prev_A = rfm_label(R_prev, F_prev, M_prev, np.median(R_prev), np.median(F_prev), np.median(M_prev))
label_curr_A = rfm_label(R_curr, F_curr, M_curr, np.median(R_curr), np.median(F_curr), np.median(M_curr))

# 方法B: 锁定上月标准
rm_p, fm_p, mm_p = np.median(R_prev), np.median(F_prev), np.median(M_prev)
label_curr_B = rfm_label(R_curr, F_curr, M_curr, rm_p, fm_p, mm_p)

print("=" * 70)
print("RFM标签边界敏感性分析 — 鲜达2000个用户")
print("=" * 70)

flipped_A = (label_prev_A != label_curr_A).sum()
flipped_B = (label_prev_A != label_curr_B).sum()
noise_flip = flipped_A - flipped_B

print(f"\n--- 方法A: 每月用当月中位数(当前做法) ---")
print(f"总翻转率: {flipped_A}/{n_users} = {flipped_A/n_users:.1%}")
print(f"\n--- 方法B: 锁定上月标准 ---")
print(f"总翻转率: {flipped_B}/{n_users} = {flipped_B/n_users:.1%}")
print(f"减少的翻转: {flipped_A - flipped_B}人 ({(flipped_A - flipped_B)/n_users:.1%})")
print(f"\n--- 翻转分解 ---")
print(f"有效翻转(行为真变了): ~{flipped_B}人 ({flipped_B/n_users:.1%})")
print(f"噪声翻转(中位数移动):  ~{noise_flip}人 ({noise_flip/n_users:.1%})")
if flipped_A > 0:
    print(f"噪声占比: {noise_flip/flipped_A:.1%}")

# 边界用户分析
for dim_name, curr_vals, med in [('R', R_curr, np.median(R_curr)),
                                   ('F', F_curr, np.median(F_curr)),
                                   ('M', M_curr, np.median(M_curr))]:
    margin = 0.10
    lower = med * (1 - margin); upper = med * (1 + margin)
    boundary = (curr_vals > lower) & (curr_vals < upper)
    print(f"\n{dim_name}边界用户(中位+/-10%): {boundary.sum()}人 ({boundary.sum()/n_users:.1%})")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
categories = ['标签不变', '有效翻转', '噪声翻转']
values_A = [n_users - flipped_A, flipped_B, noise_flip]
values_B = [n_users - flipped_B, flipped_B, 0]
x = np.arange(len(categories)); width = 0.35
ax.bar(x - width/2, values_A, width, label='方法A(各月自定中位)', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, values_B, width, label='方法B(锁定上月标准)', color='#2ecc71', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylabel('用户数'); ax.set_title('翻转来源分解: 锁定中位数消除噪声翻转')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
margins = np.linspace(0.02, 0.30, 15)
boundary_pcts = []
for margin in margins:
    b_r = (R_curr > np.median(R_curr) * (1-margin)) & (R_curr < np.median(R_curr) * (1+margin))
    b_f = (F_curr > np.median(F_curr) * (1-margin)) & (F_curr < np.median(F_curr) * (1+margin))
    b_m = (M_curr > np.median(M_curr) * (1-margin)) & (M_curr < np.median(M_curr) * (1+margin))
    boundary_pcts.append((b_r | b_f | b_m).sum() / n_users)
ax.plot(margins * 100, [p*100 for p in boundary_pcts], 'o-', linewidth=2, color='purple')
ax.set_xlabel('边界宽度(中位数+/-%)'); ax.set_ylabel('边界用户占比(%)')
ax.set_title('边界宽度 vs 边界用户占比')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rfm_boundary_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n图表已保存: rfm_boundary_sensitivity.png")

print("\n" + "=" * 70)
print("对鲜达的建议")
print("=" * 70)
print(f"1. 锁定分界标准: 用上月的中位数作为当月分界 => 消除约{noise_flip}人({noise_flip/n_users:.1%})的噪声翻转")
print("2. 边界用户(中位+/-10%)做软分群: 同时标注两个群体")
print("3. 每月自动输出'翻转来源分解图'发给运营总监")


#### 练习反馈

**参考答案**：
- 噪声翻转通常占总翻转的25-40%（取决于中位数移动幅度）
- 锁定分界标准(用上月的中位数判断本月)可消除来源B
- 边界用户(中位+/-10%)通常占15-25%

**常见偏差**：
1. "锁定标准就好了" — 锁定标准解决来源B(中位数移动)，但来源C(边界用户)仍然存在。最终解决方案=锁定标准+边界软分群+K-Means辅助。
2. "翻转就是坏事" — 如果用户真的从"新用户"升级为"重要发展"，翻转是应该的。关键是区分有效翻转和噪声翻转。
3. "边界用户占比小不用管" — 15-25%的用户在边界上——如果鲜达有300万用户，那就是45-75万人。

**补救路径**：
- 做一个自动化报告：每月自动输出"翻转来源分解图"->发给运营总监
- 对边界用户实施差异化策略：不按"高/低"发券，按"边界概率"发梯度券


---

### 概念4：RFM数据质量的三重污染——退货、拆单、大促 [防护] [必修]

#### 认知冲突

双11结束后的第二周，市场VP周总急匆匆跑来找你："RFM报告显示我们的'重要价值'用户翻了一倍——从161人涨到了320人！我们的核心用户真的翻倍了吗？"

你心中一凛。大促期间用户下单量确实暴增——但有多少是"真实的消费行为"、多少是"异常行为"？
1. **退货污染R值**：用户双11买了->双11后退货->但R值还是"双11那天"->虚假活跃
2. **拆单污染F值**：用户把一单拆成三单(凑满减)->F从1变成3->虚假高频
3. **大促污染M值**：用户一次囤了半年量->M虚高->但接下来5个月都不会买

**如果数据被污染，RFM分群就是垃圾进、垃圾出。**


#### 概念引入

**一句话白话**：RFM数据质量有三类最常见的"污染源"，每一种都能把用户分到错误的群——而且它们在大促后集中爆发。

**核心逻辑 + 具体例子**

**污染1：退货扭曲R值**
- 用户1月1日下单->1月5日收到->1月8日退货
- 如果只看订单表的order_date：R=1月1日->看起来"最近来过"
- 实际：用户已经退货了，真实的"最近有效消费"可能是2个月前
- 检测方法：用orders LEFT JOIN returns排除已退货订单->计算"净R值"

**污染2：拆单虚增F值**
- 用户为了用"满99减20"的券，把120元的东西拆成99+21两单
- 如果不合并：F=2次 -> 看起来"高频"
- 实际：同一时间、同一收货地址的多订单应该合并为一次"购买session"
- 检测方法：同一用户+/-30分钟内+同一地址的多订单->合并成1次

**污染3：大促囤货虚增M值**
- 用户双11一次性买了6个月的洗衣液(300元)->M虚高
- 如果直接用这个M值：用户被标为"重要价值"(M>=52元)
- 实际：接下来5个月这个用户都不会再买
- 检测方法：分析每个品类用户的"正常月消费量"，超过2倍标准差的标为异常

**与核心概念的关系**：[防护] — 主课讲RFM假设"数据是干净的"，但现实中数据从来不会干净。本概念提供三个具体的数据质量检测器。来自全课程永久空白区4.2"数据质量工程"。

**自包含说明**：三个检测器都只需要SQL/Pandas的基础操作（JOIN/GROUP BY/标准差），不需要额外的统计知识。


#### 在沙盘中的用武之地

鲜达的生鲜品类退货率约5-8%，大促期间拆单率可达20%，囤货行为在粮油调味品类尤其严重。如果不做数据清洗：
- 约8%的用户R值被退货"刷新"->虚假活跃
- 约20%的大促用户F值虚高->虚假忠诚
- 约15%的囤货用户M值虚高->虚假高价值
- 综合：约25-35%的用户标签可能被污染->运营券投向错误->直接浪费预算


In [ ]:
# ============================================================
# 概念4 练习：RFM数据质量三重污染检测
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(2024)
n_users = 2000

# ---- 生成"干净"的RFM ----
R_clean = np.random.exponential(25, n_users).astype(int).clip(0, 120)
F_clean = np.random.poisson(4, n_users).clip(0, 20)
M_clean = np.round(np.random.lognormal(4, 0.8, n_users), 2).clip(5, 2000)

# ---- 注入污染 ----
# 污染1: 退货(约8%)
n_returns = int(n_users * 0.08)
return_users = np.random.choice(n_users, n_returns, replace=False)
R_dirty = R_clean.copy()
for u in return_users:
    R_dirty[u] = max(1, np.random.randint(1, 5))

# 污染2: 拆单(约20%)
n_split = int(n_users * 0.20)
split_users = np.random.choice(n_users, n_split, replace=False)
F_dirty = F_clean.copy()
for u in split_users:
    F_dirty[u] = F_clean[u] + np.random.randint(1, 4)

# 污染3: 囤货(约15%)
n_stockpile = int(n_users * 0.15)
stockpile_users = np.random.choice(n_users, n_stockpile, replace=False)
M_dirty = M_clean.copy()
for u in stockpile_users:
    M_dirty[u] = M_clean[u] * np.random.uniform(2.0, 5.0)

df = pd.DataFrame({
    'user_id': range(1, n_users+1),
    'R_clean': R_clean, 'F_clean': F_clean, 'M_clean': M_clean,
    'R_dirty': R_dirty, 'F_dirty': F_dirty, 'M_dirty': M_dirty
})

# 用干净数据的中位数做分界
rm = np.median(R_clean); fm = np.median(F_clean); mm = np.median(M_clean)

def classify_rfm(R, F, M):
    rh = int(R <= rm); fh = int(F >= fm); mh = int(M >= mm)
    if rh==1 and fh==1 and mh==1: return '重要价值'
    elif rh==0 and fh==1 and mh==1: return '重要挽留'
    elif rh==1 and fh==0 and mh==1: return '重要发展'
    elif rh==0 and fh==0 and mh==1: return '流失高价值'
    elif rh==1 and fh==1 and mh==0: return '一般价值'
    elif rh==0 and fh==1 and mh==0: return '一般挽留'
    elif rh==1 and fh==0 and mh==0: return '新用户/低价值'
    else: return '流失用户'

df['label_clean'] = df.apply(lambda r: classify_rfm(r['R_clean'], r['F_clean'], r['M_clean']), axis=1)
df['label_dirty'] = df.apply(lambda r: classify_rfm(r['R_dirty'], r['F_dirty'], r['M_dirty']), axis=1)

# ---- 污染影响分析 ----
flipped = (df['label_clean'] != df['label_dirty'])
r_changed = (df['R_clean'] != df['R_dirty'])
f_changed = (df['F_clean'] != df['F_dirty'])
m_changed = (df['M_clean'] != df['M_dirty'])

print("=" * 70)
print("RFM数据质量三重污染 — 影响量化分析")
print("=" * 70)
print(f"\n总标签翻转(污染导致): {flipped.sum()}人 ({flipped.sum()/n_users:.1%})")
print(f"\n--- 污染1: 退货(R值) ---")
print(f"  R值被修改: {r_changed.sum()}人, 其中标签翻转: {(flipped & r_changed).sum()}人")
print(f"--- 污染2: 拆单(F值) ---")
print(f"  F值被修改: {f_changed.sum()}人, 其中标签翻转: {(flipped & f_changed).sum()}人")
print(f"--- 污染3: 囤货(M值) ---")
print(f"  M值被修改: {m_changed.sum()}人, 其中标签翻转: {(flipped & m_changed).sum()}人")

hv_clean = (df['label_clean'] == '重要价值').sum()
hv_dirty = (df['label_dirty'] == '重要价值').sum()
print(f"\n>>> 重要价值: 干净={hv_clean}人 -> 污染={hv_dirty}人 (虚增{hv_dirty-hv_clean}人, +{(hv_dirty-hv_clean)/hv_clean*100:.0f}%)")

# 标签分布变化
print("\n--- 标签分布变化 ---")
for label in ['重要价值','重要挽留','重要发展','流失高价值','一般价值','一般挽留','新用户/低价值','流失用户']:
    c = (df['label_clean'] == label).sum()
    d = (df['label_dirty'] == label).sum()
    if c != d:
        print(f"  {label}: {c} -> {d} ({d-c:+d})")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
labels = ['重要价值', '重要挽留', '重要发展', '流失高价值',
          '一般价值', '一般挽留', '新用户/低价值', '流失用户']
x = np.arange(len(labels)); width = 0.35
clean_counts = [(df['label_clean'] == l).sum() for l in labels]
dirty_counts = [(df['label_dirty'] == l).sum() for l in labels]
ax.bar(x - width/2, clean_counts, width, label='干净数据', color='#2ecc71', alpha=0.8)
ax.bar(x + width/2, dirty_counts, width, label='污染数据', color='#e74c3c', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('用户数'); ax.set_title('数据污染对RFM标签分布的影响')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
sample_n = min(500, n_users)
sample_idx = np.random.choice(n_users, sample_n, replace=False)
ax.scatter(df.loc[sample_idx, 'M_clean'], df.loc[sample_idx, 'M_dirty'],
           c='gray', alpha=0.3, s=10)
stockpile_sample = np.intersect1d(stockpile_users, sample_idx)
ax.scatter(df.loc[stockpile_sample, 'M_clean'], df.loc[stockpile_sample, 'M_dirty'],
           c='red', alpha=0.7, s=20, label=f'囤货({n_stockpile}人)')
max_m = max(df['M_clean'].max(), df['M_dirty'].max())
ax.plot([0, max_m], [0, max_m], '--', color='green', alpha=0.5, label='M_dirty=M_clean')
ax.set_xlabel('干净M值(元)'); ax.set_ylabel('污染M值(元)')
ax.set_title('囤货污染: M值系统偏高'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
contamination_types = ['退货(R污染)', '拆单(F污染)', '囤货(M污染)', '三重叠加']
flip_rates = []
for mask in [r_changed, f_changed, m_changed]:
    affected = df[mask]
    flip_rates.append((affected['label_clean'] != affected['label_dirty']).mean() if len(affected) > 0 else 0)
triple = r_changed | f_changed | m_changed
affected_all = df[triple]
flip_rates.append((affected_all['label_clean'] != affected_all['label_dirty']).mean() if len(affected_all) > 0 else 0)
bars = ax.bar(contamination_types, [r*100 for r in flip_rates],
              color=['#e74c3c', '#f39c12', '#9b59b6', '#2c3e50'])
ax.set_ylabel('标签翻转率(%)'); ax.set_title('各污染源导致的标签翻转率')
for bar, rate in zip(bars, flip_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.1%}', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('rfm_data_quality_contamination.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n图表已保存: rfm_data_quality_contamination.png")
print("\n对鲜达的清洁建议:")
print("1. 计算RFM前必须排除已全额退货的订单")
print("2. 同用户+/-30分钟内同地址的多订单->合并为1次购买session")
print("3. 大促月(6/11月)的M值用非促月均消费做校准")
print("4. 每期RFM更新前自动跑以上3个检测->污染率>10%则告警")


#### 练习反馈

**参考答案**：
- 三重污染综合导致约25-35%的用户标签发生错误
- 退货污染R值->虚假活跃(约8%用户)
- 拆单虚增F值->虚假高频(约20%用户)
- 囤货虚增M值->虚假高价值(约15%用户)
- "重要价值"用户可能虚增30-50%

**常见偏差**：
1. "数据脏一点没关系，大体方向对就行" — 如果25%的用户标签是错的，那运营团队每花4块钱就有1块钱投错了人。年预算1000万=浪费250万。
2. "只看退货就行" — 退货是最容易检测的(有退货记录)，但拆单和囤货更隐蔽——它们没有逆转记录，需要主动检测。
3. "数据清洗做完一次就行了" — 污染源会进化（新的促销玩法、新的退货政策），清洗规则需要持续更新。

**补救路径**：
- 建立"RFM数据质量周检"：每周自动运行三个检测器->输出污染率报告
- 大促后强制数据清洗：双11/618后第一周不对RFM标签做重大运营决策
- 长期：用Pandera/Great Expectations自动验证RFM数据schema


---

### 概念5：RFM完整SQL生产实现（含迁移矩阵）[落地] [必修]

#### 认知冲突

孙工冲进你的工位："看，这是现在每周五跑RFM的流程。"他把屏幕转过来——一个Python脚本，从MySQL导出CSV->pandas计算->写回Excel->邮件发给运营。"300万用户光导出就要15分钟，pandas算RFM再把数据写回数据库又15分钟。每周五下午我就做这一件事。"

你知道如果鲜达用户涨到500万或1000万，这个Python脚本迟早崩。**真正的生产环境RFM，应该全部用SQL完成——从原始订单到标签表，一条SQL流水线。**

主课所有的RFM代码都是Python/pandas写的。**但实际工作中，超过10万用户就应该用SQL。** 来自全课程永久空白区2.1"SQL高级模式"。


#### 概念引入

**一句话白话**：RFM的SQL实现就是把中位数分界、标签映射、迁移矩阵全部写成SQL——数据库里跑，不需要导出数据。

**核心逻辑 + 具体例子**

RFM的SQL流水线分5步（每步一个CTE或临时表）：

**Step 1: 计算每用户的R/F/M值**
```sql
WITH user_rfm AS (
    SELECT
        user_id,
        DATEDIFF(CURRENT_DATE, MAX(order_date)) AS R,
        COUNT(DISTINCT order_id) AS F,
        SUM(amount) AS M
    FROM orders
    WHERE order_date >= DATE_SUB(CURRENT_DATE, INTERVAL 90 DAY)
      AND is_returned = 0  -- 排除退货!
    GROUP BY user_id
)
```

**Step 2: 计算全局中位数**
```sql
-- 用PERCENTILE_CONT或ROW_NUMBER取中位数
-- MySQL没有PERCENTILE_CONT，用ROW_NUMBER技巧
```

**Step 3: 映射到8个群体标签**
```sql
CASE
    WHEN R <= r_median AND F >= f_median AND M >= m_median THEN '重要价值'
    WHEN R > r_median AND F >= f_median AND M >= m_median THEN '重要挽留'
    ...
END AS rfm_segment
```

**Step 4: 保存历史标签（用于迁移矩阵）**
```sql
INSERT INTO rfm_label_history (user_id, rfm_segment, r_value, f_value, m_value, snapshot_date)
SELECT user_id, rfm_segment, R, F, M, CURRENT_DATE FROM user_rfm_labeled;
```

**Step 5: 计算迁移矩阵**
```sql
SELECT
    prev.rfm_segment AS prev_segment,
    curr.rfm_segment AS curr_segment,
    COUNT(*) AS user_count
FROM rfm_label_history prev
JOIN rfm_label_history curr
    ON prev.user_id = curr.user_id
    AND prev.snapshot_date = '2026-06-01'
    AND curr.snapshot_date = '2026-07-01'
GROUP BY 1, 2;
```

**与核心概念的关系**：[落地] — 主课教了"用Python算RFM"，本概念教"用SQL算RFM（生产环境）"。Python适合探索分析，SQL适合生产流水线。

**自包含说明**：所有SQL都使用标准SQL语法（兼容MySQL/PostgreSQL/Hive），只需要基础的GROUP BY/JOIN/CASE WHEN知识，这些W5会深入。


#### 在沙盘中的用武之地

鲜达的用户量300万->孙工每周五手工跑Python脚本。全SQL方案的优势：
1. **速度**：数据库内计算，不需要导出300万行数据到Python
2. **可靠性**：SQL定时任务(Crontab/Airflow)自动执行，不需要人操作
3. **可追溯**：每月标签存入历史表->任何时间点都可以重建迁移矩阵
4. **可扩展**：1000万用户也能跑（加索引+分区后）


In [ ]:
-- ============================================================
-- 概念5 练习：RFM完整SQL实现（MySQL语法）
-- ============================================================
-- 提示：以下SQL需要在真实MySQL数据库中执行
-- 这里提供的是完整的可运行SQL代码，你可以复制到你的数据库测试

-- ============ Step 1: 计算每用户RFM值 ============
-- 假设orders表结构: order_id, user_id, order_date, amount, is_returned
/*
CREATE OR REPLACE VIEW user_rfm_values AS
SELECT
    user_id,
    DATEDIFF(CURRENT_DATE, MAX(order_date)) AS R_days,
    COUNT(DISTINCT order_id) AS F_count,
    SUM(amount) AS M_amount
FROM orders
WHERE order_date >= DATE_SUB(CURRENT_DATE, INTERVAL 90 DAY)
  AND is_returned = 0  -- 关键: 排除退货订单!
GROUP BY user_id
HAVING COUNT(DISTINCT order_id) > 0;
*/

-- ============ Step 2: 计算全局中位数（MySQL 8.0+ 窗口函数版） ============
/*
CREATE OR REPLACE VIEW rfm_medians AS
SELECT
    -- R中位数: 用ROW_NUMBER定位中间行
    AVG(CASE
        WHEN rn_r BETWEEN FLOOR((total + 1) / 2) AND CEIL((total + 1) / 2)
        THEN R_days END
    ) AS R_median,
    AVG(CASE
        WHEN rn_f BETWEEN FLOOR((total + 1) / 2) AND CEIL((total + 1) / 2)
        THEN F_count END
    ) AS F_median,
    AVG(CASE
        WHEN rn_m BETWEEN FLOOR((total + 1) / 2) AND CEIL((total + 1) / 2)
        THEN M_amount END
    ) AS M_median
FROM (
    SELECT
        R_days, F_count, M_amount,
        ROW_NUMBER() OVER (ORDER BY R_days) AS rn_r,
        ROW_NUMBER() OVER (ORDER BY F_count) AS rn_f,
        ROW_NUMBER() OVER (ORDER BY M_amount) AS rn_m,
        COUNT(*) OVER () AS total
    FROM user_rfm_values
) t;
*/

-- ============ Step 3: 打标签 ============
/*
CREATE OR REPLACE VIEW user_rfm_labeled AS
SELECT
    u.user_id,
    u.R_days, u.F_count, u.M_amount,
    CASE
        WHEN u.R_days <= m.R_median AND u.F_count >= m.F_median AND u.M_amount >= m.M_median
            THEN '重要价值'
        WHEN u.R_days >  m.R_median AND u.F_count >= m.F_median AND u.M_amount >= m.M_median
            THEN '重要挽留'
        WHEN u.R_days <= m.R_median AND u.F_count <  m.F_median AND u.M_amount >= m.M_median
            THEN '重要发展'
        WHEN u.R_days >  m.R_median AND u.F_count <  m.F_median AND u.M_amount >= m.M_median
            THEN '流失高价值'
        WHEN u.R_days <= m.R_median AND u.F_count >= m.F_median AND u.M_amount <  m.M_median
            THEN '一般价值'
        WHEN u.R_days >  m.R_median AND u.F_count >= m.F_median AND u.M_amount <  m.M_median
            THEN '一般挽留'
        WHEN u.R_days <= m.R_median AND u.F_count <  m.F_median AND u.M_amount <  m.M_median
            THEN '新用户/低价值'
        ELSE '流失用户'
    END AS rfm_segment,
    CURRENT_DATE AS snapshot_date
FROM user_rfm_values u
CROSS JOIN rfm_medians m;
*/

-- ============ Step 4: 保存历史快照 ============
/*
INSERT INTO rfm_label_history (user_id, rfm_segment, R_days, F_count, M_amount, snapshot_date)
SELECT user_id, rfm_segment, R_days, F_count, M_amount, snapshot_date
FROM user_rfm_labeled;
*/

-- ============ Step 5: 迁移矩阵 ============
/*
SELECT
    prev.rfm_segment AS prev_segment,
    curr.rfm_segment AS curr_segment,
    COUNT(*) AS user_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY prev.rfm_segment), 1) AS pct
FROM rfm_label_history prev
JOIN rfm_label_history curr
    ON prev.user_id = curr.user_id
WHERE prev.snapshot_date = '2026-06-01'
  AND curr.snapshot_date = '2026-07-01'
GROUP BY 1, 2
ORDER BY 1, 2;
*/

-- ============ Step 6: 自动识别流失预警用户 ============
/*
SELECT
    prev.user_id,
    prev.rfm_segment AS prev_segment,
    curr.rfm_segment AS curr_segment,
    CASE
        WHEN prev.rfm_segment = '重要价值' AND curr.rfm_segment = '重要挽留'
            THEN 'HIGH - 高价值用户正在流失!'
        WHEN prev.rfm_segment = '重要价值' AND curr.rfm_segment = '一般价值'
            THEN 'MEDIUM - 高价值用户消费降级'
        WHEN prev.rfm_segment = '重要发展' AND curr.rfm_segment IN ('流失高价值','流失用户')
            THEN 'MEDIUM - 新增大额用户流失'
        ELSE 'LOW'
    END AS alert_level
FROM rfm_label_history prev
JOIN rfm_label_history curr
    ON prev.user_id = curr.user_id
    AND prev.snapshot_date = '2026-06-01'
    AND curr.snapshot_date = '2026-07-01'
WHERE prev.rfm_segment != curr.rfm_segment
  AND (prev.rfm_segment IN ('重要价值', '重要发展')
       OR curr.rfm_segment IN ('流失用户', '流失高价值'));
*/

print("=" * 70)
print("概念5: RFM完整SQL生产实现")
print("=" * 70)
print("以上SQL代码可直接在MySQL 8.0+环境中执行")
print("关键设计决策:")
print("1. 用ROW_NUMBER()取中位数 - MySQL没有PERCENTILE_CONT")
print("2. 中位数存为视图 - 复用历史标准(对应概念3的'锁定标准')")
print("3. 历史快照表 - 保留每期标签(对应概念3的迁移矩阵)")
print("4. 自动告警 - 高危迁移路径自动标记")
print("5. 排除退货 - WHERE is_returned=0 (对应概念4的数据清洗)")

print("\n=== 性能建议 ===")
print("- <10万用户: 以上SQL直接跑, 约5秒")
print("- 10-100万: 加索引 CREATE INDEX idx_orders_date_user ON orders(order_date, user_id)")
print("- >100万: 用Spark SQL/Hive, orders表按月份分区")
print("- 定时执行: Airflow DAG 每周一凌晨3点跑")


#### 练习反馈

**参考答案**：以上SQL是完整的RFM生产流水线——从原始订单到告警用户名单。可以在MySQL 8.0+中直接执行。

**常见偏差**：
1. "用Python就够了" — 300万用户(Python慢但能跑) vs 3000万用户(Python直接OOM)。SQL在数据库内计算，不会把数据拉出来。
2. "中位数用AVG就行了" — 如果分布严重偏斜，均值 != 中位数。用ROW_NUMBER()取中位数位置更准确。
3. "不需要history表" — 没有history表就没有迁移矩阵——RFM变成一次性快照，失去了最强大的流失预警能力。

**补救路径**：
- MySQL用户：确认版本>=8.0（窗口函数支持）
- PostgreSQL用户：用PERCENTILE_CONT(0.5)直接取中位数
- Hive/Spark用户：用PERCENTILE_APPROX(0.5)近似中位数（大数据场景）


---

### 概念6：客户健康度评分(Customer Health Score)——当RFM不够用的时候 [替代] [选修]

#### 认知冲突

鲜达最近上线了"鲜达会员"——用户支付年费享免运费+专属折扣。赵总发现一批"流失用户"(RFM三个维度都低)——但他们的会员即将到期，且最近7天每天都打开App浏览。按RFM他们是"不值得投入"的群体，但按"健康度"——他们可能只是在"货比三家"、随时可能回来。

**纯RFM的盲区：只看交易，不看行为。一个用户可能3个月没买(R高F低M低)但每天都打开App看——他不是流失了，他只是在犹豫。**


#### 概念引入

**一句话白话**：客户健康度评分是在RFM的交易维度之外，加上产品使用行为、客服互动、满意度反馈等维度，综合判断一个用户是"健康"还是"病了"。

**核心逻辑 + 具体例子**

RFM vs Health Score的覆盖维度对比：

| 维度 | RFM | Health Score | 说明 |
|------|-----|-------------|------|
| 交易活跃度 | R(最近购买) | + 最近浏览/加购/搜索 | 不买但看=可能在比价 |
| 交易频率 | F(购买次数) | + 登录频率/功能使用频率 | 天天登录但不买=对产品有兴趣但没被转化 |
| 交易价值 | M(消费金额) | + 客单价趋势/品类宽度 | M下降但品类在拓宽=用户在探索 |
| 产品使用 | - | 核心功能使用深度 | 订阅/SaaS独有的维度 |
| 客户反馈 | - | NPS/CSAT/投诉次数/客服咨询 | 行为+态度=完整画像 |
| 社交传播 | - | 推荐次数/分享次数 | 不买但推荐=品牌拥护者 |

在鲜达的场景中：
- RFM说"流失用户" -> 但用户每天打开App浏览生鲜 -> Health Score说"活跃浏览者"
- RFM说"重要价值" -> 但用户最近5次客服投诉、NPS=2 -> Health Score说"高危重要用户"
- Health Score = RFM(50%) + 产品使用(25%) + 客户反馈(15%) + 社交传播(10%)

**每个维度可以从0-100打分，加权求和得到综合健康分。**

**与核心概念的关系**：[替代] — RFM适合纯交易场景（电商零售），Health Score适合有产品使用行为的场景（SaaS/订阅/内容App）。鲜达作为"电商+SaaS化会员"的混合模式，需要两者配合。来自全课程永久空白区5.3"指标设计深度补充"。

> **不是替代RFM，而是扩充RFM**：RFM依然是Health Score的核心组件（占50%+权重），但补充产品使用和反馈维度后，能捕捉RFM捕捉不到的"行为先行指标"。


#### 在沙盘中的用武之地

鲜达的300万用户中：
- 通过Health Score，你发现15%的"流失用户"其实是"高活跃浏览者"——他们每天打开App但3个月没下单。运营策略不是"放弃"而是"定向优惠+push推荐"。
- 8%的"重要价值"用户Health Score<50——因为最近2周客服投诉激增。在你不知道的情况下，"VIP服务"的质量出了问题。
- 综合来看，Health Score让你把RFM的8个静态标签变成**动态健康信号**——不只看"过去做了什么"，还看"正在做什么"。


In [ ]:
# ============================================================
# 概念6 练习：客户健康度评分 vs RFM
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(88)
n_users = 1500

# 模拟鲜达的用户数据（含RFM + 行为 + 反馈）
df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    # RFM维度
    'R': np.random.exponential(25, n_users).astype(int).clip(0, 120),
    'F': np.random.poisson(4, n_users).clip(0, 20),
    'M': np.round(np.random.lognormal(4, 0.8, n_users), 2).clip(5, 2000),
    # 行为维度（0-100）
    'browse_freq': np.random.beta(2, 5, n_users),      # 浏览频率
    'search_depth': np.random.beta(1.5, 4, n_users),    # 搜索深度
    'cart_rate': np.random.beta(3, 7, n_users),         # 加购率
    # 反馈维度（0-100）
    'nps': np.random.normal(50, 25, n_users).clip(0, 100),
    'complaints': np.random.poisson(0.5, n_users),
    'cs_contacts': np.random.poisson(1, n_users),
})

# ---- 计算RFM标签 ----
rm = df['R'].median(); fm = df['F'].median(); mm = df['M'].median()
def rfm_label(r, f, m):
    rh = int(r <= rm); fh = int(f >= fm); mh = int(m >= mm)
    if rh and fh and mh: return '重要价值'
    elif not rh and fh and mh: return '重要挽留'
    elif rh and not fh and mh: return '重要发展'
    elif not rh and not fh and mh: return '流失高价值'
    elif rh and fh and not mh: return '一般价值'
    elif not rh and fh and not mh: return '一般挽留'
    elif rh and not fh and not mh: return '新用户/低价值'
    else: return '流失用户'
df['rfm'] = df.apply(lambda r: rfm_label(r['R'], r['F'], r['M']), axis=1)

# ---- 计算Health Score ----
# RFM分数(0-100): R越低越好、F/M越高越好
df['r_score'] = 100 * (1 - df['R'] / df['R'].max())
df['f_score'] = 100 * (df['F'] / df['F'].max())
df['m_score'] = 100 * (df['M'] / df['M'].max())
df['rfm_score'] = (df['r_score'] * 0.4 + df['f_score'] * 0.3 + df['m_score'] * 0.3)

# 行为分数(0-100)
df['behavior_score'] = (df['browse_freq'] * 40 + df['search_depth'] * 30 + df['cart_rate'] * 30)

# 反馈分数(0-100): NPS高好，投诉少好
df['feedback_score'] = (df['nps'] * 0.6 + (100 - df['complaints'] * 20).clip(0, 100) * 0.2
                         + (100 - df['cs_contacts'] * 10).clip(0, 100) * 0.2)

# 综合Health Score
df['health_score'] = (df['rfm_score'] * 0.50 + df['behavior_score'] * 0.25
                      + df['feedback_score'] * 0.15 + 10)  # 社交传播暂无数据

df['health_score'] = df['health_score'].clip(0, 100).round(0).astype(int)

# ---- 分析：RFM vs Health Score 的差异 ----
print("=" * 70)
print("客户健康度评分 vs RFM — 鲜达1500个用户")
print("=" * 70)

# RFM"流失用户"中Health Score高的
lost_users = df[df['rfm'] == '流失用户']
high_health_lost = lost_users[lost_users['health_score'] > 40]
print(f"\nRFM='流失用户'({len(lost_users)}人)中:")
print(f"  Health Score>40: {len(high_health_lost)}人 ({len(high_health_lost)/len(lost_users):.0%})")
print(f"  -> 这些用户虽然不买，但行为和反馈表明他们还在活跃!不应该被'放弃'")

# RFM"重要价值"中Health Score低的
hv_users = df[df['rfm'] == '重要价值']
low_health_hv = hv_users[hv_users['health_score'] < 50]
print(f"\nRFM='重要价值'({len(hv_users)}人)中:")
print(f"  Health Score<50: {len(low_health_hv)}人 ({len(low_health_hv)/len(hv_users):.0%})")
print(f"  -> 高价值用户中有人在'生病'!需要主动关怀而非坐等流失")

# 各RFM群体的平均Health Score
print("\n--- 各RFM群体的平均Health Score ---")
for seg in ['重要价值','重要挽留','重要发展','流失高价值','一般价值','一般挽留','新用户/低价值','流失用户']:
    seg_data = df[df['rfm'] == seg]
    if len(seg_data) > 0:
        print(f"  {seg}: Health Score均值={seg_data['health_score'].mean():.0f}, "
              f"RFM得分={seg_data['rfm_score'].mean():.0f}, "
              f"行为得分={seg_data['behavior_score'].mean():.0f}, "
              f"反馈得分={seg_data['feedback_score'].mean():.0f}")

# 可视化: 二维矩阵
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
segments = ['重要价值','重要发展','一般价值','一般挽留','重要挽留','流失高价值','新用户/低价值','流失用户']
colors_list = ['#e74c3c','#f1c40f','#2ecc71','#1abc9c','#e67e22','#9b59b6','#3498db','#95a5a6']
for seg, c in zip(segments, colors_list):
    mask = df['rfm'] == seg
    if mask.sum() > 0:
        ax.scatter(df.loc[mask, 'rfm_score'], df.loc[mask, 'health_score'],
                   c=c, label=seg, alpha=0.5, s=15)
ax.plot([0, 100], [0, 100], '--', color='gray', alpha=0.5, label='Health=RFM')
ax.set_xlabel('RFM得分'); ax.set_ylabel('Health Score')
ax.set_title('RFM得分 vs Health Score: 行为+反馈改变了用户排序')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = axes[1]
ranges = [(0,30), (30,50), (50,70), (70,100)]
health_labels = ['不健康(0-30)', '需关注(30-50)', '健康(50-70)', '非常健康(70-100)']
colors_health = ['#e74c3c', '#f39c12', '#2ecc71', '#27ae60']
for (lo, hi), label, c in zip(ranges, health_labels, colors_health):
    mask = (df['health_score'] >= lo) & (df['health_score'] < hi)
    ax.bar(label, mask.sum(), color=c, alpha=0.8)
    ax.text(label, mask.sum() + 5, f'{mask.sum()}人\n({mask.sum()/n_users:.0%})', ha='center')
ax.set_ylabel('用户数'); ax.set_title('Health Score分布 (鲜达1500个用户)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('health_score_vs_rfm.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n图表已保存: health_score_vs_rfm.png")
print("\n>>> 关键结论: Health Score补充了RFM缺失的行为+反馈维度")
print("    -> RFM说'放弃'但Health Score说'有救' -> 重新评估运营策略")


#### 练习反馈

**参考答案**：
- RFM标记的"流失用户"中，约30-40%在Health Score中表现良好(行为活跃+反馈好)
- RFM标记的"重要价值"中，约10-15%在Health Score中表现差(投诉多+行为下降)
- Health Score让你发现RFM看不到的两个群体：高潜力非买家、高风险高价值用户

**常见偏差**：
1. "Health Score替代RFM" — 不是替代，是补充。RFM(交易维度)仍然是Health Score的核心组件(占50%+权重)。没有RFM的Health Score就像没有骨架的身体。
2. "维度越多越好" — 维度多了权重就难定。建议从3-4个维度开始，逐步增加。每增加一个维度问自己：这个维度告诉了我RFM没告诉我的什么？
3. "权重拍脑袋" — 初始权重可以用业务经验(strategy部门讨论)，每季度用历史数据回溯优化(看哪个维度的权重对预测流失最重要)。

**补救路径**：
- 想自动化Health Score -> 用加权公式写入SQL，跟RFM标签一起更新
- 想验证Health Score好不好 -> 回溯6个月，看Health Score低的用户是否真的流失了更多


---

### 概念7：RFM产品化——指标身份证 + SPC统计过程监控 [落地] [选修]

#### 认知冲突

三周后，你完成了K-Means探索(k=5更优)、数据清洗流水线、SQL自动化、甚至Health Score的草稿。赵总看了很满意，但问了一个让你语塞的问题：

> "这些都很好——但下个月如果我不在了，谁来维护这套东西？如果有人改了RFM计算逻辑但没告诉你，你怎么知道？如果用户行为发生了结构性变化(比如疫情后大家都转线上)，RFM的分界标准会自动调整吗？"

你意识到：**你把RFM从一个"周报"升级成了"系统"，但你没有给这个系统写"使用说明书"和"健康检查"。** 来自全课程永久空白区5.3"指标身份证10字段模板"和"指标腐烂生命周期"。


#### 概念引入

**一句话白话**：RFM产品化 = 给RFM系统写"身份证"(谁定义的？怎么算的？更新频率？) + "体检报告"(今天的RFM分布正常吗？趋势在恶化吗？)。

**核心逻辑**

**A. 指标身份证 (Metric ID Card)** — 10字段模板：

| # | 字段 | RFM示例 |
|---|------|--------|
| 1 | 指标名称 | RFM用户分群标签 |
| 2 | 中文名 | 基于Recency/Frequency/Monetary的用户价值分层 |
| 3 | 定义 | R=最近一次有效消费距今天数; F=90天内有效消费次数; M=90天内有效消费总额 |
| 4 | 公式 | R<=P50->R_high=1; F>=P50->F_high=1; M>=P50->M_high=1; 2x2x2交叉出8群 |
| 5 | 数据来源 | orders表(排除is_returned=1), order_items表(排除is_promotion拆单) |
| 6 | 更新频率 | 每周一凌晨3点(UTC+8) |
| 7 | 责任人 | 数据运营团队(owner: 你的名字), 运营团队(consumer) |
| 8 | 已知问题 | 大促月(6/11)M值偏高; 拆单检测仍在优化; 边界用户~20%标签欠稳定 |
| 9 | SLA | 每周一8:00前完成更新; 污染率>10%需在周三前修复 |
| 10 | 版本 | v2.1 (2026-07-01) — 新增数据清洗流水线, 锁定上月分界标准 |

**B. SPC统计过程监控** — 让RFM自己"体检"：

SPC(Statistical Process Control)来自制造业——用控制图监控生产线的稳定性。应用到RFM：
1. **基线期**（如过去12周）：计算每期RFM群体分布的均值和标准差
2. **控制限**：均值 +/- 3倍标准差 = "正常范围"。超出=异常信号。
3. **每周自动检测**：本期各群体占比是否在控制限内？
4. **告警**：某群体占比连续3周超出控制限 = 系统性变化(非随机波动)

在鲜达的应用：如果"重要价值"群体占比连续3周低于控制下限 -> 自动告警 -> 触发根因分析：是竞品挖走了核心用户？是产品质量出了问题？还是中位数标准需要重新校准？

**C. 指标腐烂生命周期** — RFM也会"过期"：

阶段1(健康): RFM准确反映用户价值，运营团队信任标签
阶段2(被博弈): 运营开始"刷"RFM——给快流失的用户塞优惠券让他们回来买一次->R值被"买"回来
阶段3(失真): 用户行为结构性变化(如疫情后生鲜线上渗透率翻倍)->旧的P50标准不再适用
阶段4(失效): RFM标签变成了"橡皮图章"——没人再根据它做决策
阶段5(替换): 团队转向新模型(Health Score / ML分群)

**防腐蚀策略**：
- 每季度review一次P50分界标准（是否需要重新校准）
- 监控"标签-策略匹配率"（运营团队实际按标签执行的比例）
- 建立"RFM vs Health Score"双轨制——当RFM失效时Health Score顶上

**与核心概念的关系**：[落地] — 把前面6个概念整合成一个"可持续运行的系统"。没有产品化，再好的分析方法也会在3个月后被遗忘。来自全课程永久空白区5.3。


#### 在沙盘中的用武之地

鲜达的RFM系统经过你的升级后：
1. 每周一早上8点，赵总打开看板 -> 看到最新的RFM群体分布 + 迁移趋势 + SPC告警
2. 如果有异常(如"重要价值"跌出控制限) -> 自动发钉钉/企微告警
3. 孙工不再需要每周五下午跑脚本 -> 他把时间花在更有价值的事情上(如分析告警原因)
4. 新同事入职 -> 打开"RFM指标身份证"就知道这个标签怎么来的
5. 季度review -> 看SPC趋势图，判断是否需要重新校准P50


In [ ]:
# ============================================================
# 概念7 练习：RFM SPC统计过程监控
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(123)

# 模拟鲜达过去24周的"重要价值"占比数据
weeks = list(range(1, 25))
# 前16周基线: 均值16%, 标准差2%
baseline = np.random.normal(0.16, 0.02, 16)
# 第17-20周: 开始下降(竞品上线)
decline = np.array([0.15, 0.14, 0.12, 0.11])
# 第21-24周: 恢复
recovery = np.array([0.12, 0.13, 0.15, 0.16])
hv_pct = np.concatenate([baseline, decline, recovery])
hv_pct = np.clip(hv_pct, 0.05, 0.25)

df_spc = pd.DataFrame({'week': weeks, 'hv_pct': hv_pct})

# SPC计算(用前16周做基线)
baseline_mean = baseline.mean()
baseline_std = baseline.std()
ucl = baseline_mean + 3 * baseline_std  # 上控制限
lcl = baseline_mean - 3 * baseline_std  # 下控制限
ucl2 = baseline_mean + 2 * baseline_std  # 上警戒限
lcl2 = baseline_mean - 2 * baseline_std  # 下警戒限

print("=" * 70)
print("RFM SPC统计过程监控 — 鲜达24周'重要价值'占比")
print("=" * 70)
print(f"基线(前16周): 均值={baseline_mean:.1%}, 标准差={baseline_std:.2%}")
print(f"控制限: [{lcl:.1%}, {ucl:.1%}] (均值+/-3sigma)")
print(f"警戒限: [{lcl2:.1%}, {ucl2:.1%}] (均值+/-2sigma)")

# 检测异常
df_spc['out_of_control'] = (df_spc['hv_pct'] < lcl) | (df_spc['hv_pct'] > ucl)
df_spc['warning'] = ((df_spc['hv_pct'] < lcl2) | (df_spc['hv_pct'] > ucl2)) & ~df_spc['out_of_control']

alerts = df_spc[df_spc['out_of_control'] | df_spc['warning']]
print(f"\n告警汇总:")
for _, row in alerts.iterrows():
    level = 'HIGH' if row['out_of_control'] else 'WARNING'
    print(f"  第{int(row['week'])}周: {row['hv_pct']:.1%} - [{level}]")

# 检测连续下降(连续3周低于均值-1sigma)
df_spc['below_1sigma'] = df_spc['hv_pct'] < (baseline_mean - baseline_std)
decline_streak = 0
for i, row in df_spc.iterrows():
    if row['below_1sigma']:
        decline_streak += 1
    else:
        decline_streak = 0
    if decline_streak >= 3:
        print(f"  第{int(row['week'])}周: 连续{decline_streak}周低于均值-1sigma -> 系统性下降!触发根因分析")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

ax = axes[0]
ax.plot(df_spc['week'], df_spc['hv_pct'] * 100, 'o-', linewidth=2, color='#2c3e50', label='重要价值占比')
ax.axhline(y=baseline_mean * 100, color='green', linestyle='-', linewidth=1.5, label=f'基线均值({baseline_mean:.1%})')
ax.axhline(y=ucl * 100, color='red', linestyle='--', linewidth=1, label=f'UCL({ucl:.1%})')
ax.axhline(y=lcl * 100, color='red', linestyle='--', linewidth=1, label=f'LCL({lcl:.1%})')
ax.axhline(y=ucl2 * 100, color='orange', linestyle=':', linewidth=1, alpha=0.7, label=f'+2sigma')
ax.axhline(y=lcl2 * 100, color='orange', linestyle=':', linewidth=1, alpha=0.7, label=f'-2sigma')
ax.axvspan(16.5, 20.5, alpha=0.1, color='red', label='下降期(竞品上线)')
ax.fill_between(df_spc['week'], lcl * 100, ucl * 100, alpha=0.05, color='gray')

# 标注异常点
for _, row in alerts.iterrows():
    color = 'red' if row['out_of_control'] else 'orange'
    ax.annotate(f'{row["hv_pct"]:.1%}', (row['week'], row['hv_pct'] * 100),
                textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9, color=color)

ax.set_xlabel('周'); ax.set_ylabel('重要价值占比(%)')
ax.set_title('RFM SPC控制图: 重要价值群体占比监控')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
# 8个群体的完整SPC监测panels
segments = ['重要价值', '重要挽留', '重要发展', '流失高价值',
            '一般价值', '一般挽留', '新用户/低价值', '流失用户']
ax.axis('off')
table_data = [['群体', '基线均值', '当前值', '趋势', '状态']]
np.random.seed(456)
for seg in segments:
    mean_pct = np.random.uniform(0.08, 0.18)
    current = mean_pct + np.random.uniform(-0.03, 0.03)
    trend_val = current - mean_pct
    trend = 'up' if trend_val > 0.01 else ('down' if trend_val < -0.01 else 'stable')
    status = 'NORMAL' if abs(trend_val) < 0.02 else ('WARNING' if abs(trend_val) < 0.04 else 'ALERT')
    table_data.append([seg, f'{mean_pct:.1%}', f'{current:.1%}', trend, status])

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.18, 0.15, 0.15, 0.12, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)
# 颜色标注
for i in range(1, len(table_data)):
    status = table_data[i][4]
    color = '#e74c3c' if status == 'ALERT' else ('#f39c12' if status == 'WARNING' else '#2ecc71')
    table[(i, 4)].set_facecolor(color)
    table[(i, 4)].set_text_props(color='white', fontweight='bold')
ax.set_title('RFM 8群体SPC监测面板 (本周)', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('rfm_spc_monitoring.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n图表已保存: rfm_spc_monitoring.png")
print("\n>>> SPC把RFM从'一次性分析'升级为'持续监控系统'")


#### 练习反馈

**参考答案**：
- SPC控制图在第18-20周检测到"重要价值"占比跌破LCL -> 触发告警
- 连续3周下降规则在第19周触发 -> 建议24小时内启动根因分析
- 第21-24周恢复 -> 证明干预有效（或竞品影响消退）

**常见偏差**：
1. "SPC告警=一定有事故" — SPC只告诉你有异常信号，不告诉原因。需要人工判断是"真异常"还是"正常波动"（3sigma下仍有0.3%的误报率）。
2. "控制限设了就永远不变" — 当用户行为发生结构性变化(如疫情后线上渗透率翻倍)，需要重新校准基线。建议每季度review一次。
3. "只看一个群体" — 需要看全部8个群体。如果"重要价值"下降但"重要发展"上升 -> 可能只是用户在成长，不是流失。

**补救路径**：
- 想在生产环境部署SPC -> 写一个每周自动运行的SQL/Python脚本，输出SPC报告
- 想更进一步 -> 加入Western Electric Rules(制造业的8条异常检测规则)，提升检测灵敏度


---

# 第四部分：综合沙盘演练（35分钟）

回到鲜达的沙盘场景。现在你有7个新工具，来完成赵总交给你的任务。


### 题1：多概念协同 — 完整RFM系统升级方案

**场景**：赵总要你在两周后的月度复盘会上展示"RFM系统升级方案"，说服三位VP采用新系统。

**任务**：综合使用K-Means(概念1)、轮廓系数(概念2)、边界敏感性(概念3)、数据清洗(概念4)四个概念，在脑海中为鲜达设计一份完整的升级方案。要求：
1. 用K-Means+轮廓系数证明：最优k=5还是k=8？为什么？
2. 用边界敏感性分析：当前中位数法的噪声翻转率是多少？锁定标准后能降低多少？
3. 用数据清洗检测：大促后数据被污染了多少？清洗后"重要价值"的变化是多少？
4. 给出从"当前做法"到"升级方案"的迁移路径（分3个阶段）

<details><summary>点击查看解题框架</summary>

**第一阶段（第1周）：诊断与验证**
- 跑K-Means(k=2..15) + 轮廓系数 -> 发现最优k≈5
- 跑边界敏感性分析 -> 噪声翻转率约30%
- 跑数据清洗检测 -> 大促后"重要价值"虚增约40%
- 产出：《RFM系统诊断报告》-> 给赵总看"当前系统的问题量化"

**第二阶段（第2周）：灰度测试**
- 对10%用户用k=5+分位数分箱（兼顾K-Means的洞察和运营可解释性）
- 锁定上月P50标准 -> 消除噪声翻转
- 上线数据清洗流水线（排除退货/合并拆单/校准囤货）
- 产出：《灰度测试对比报告》-> 新旧系统在10%用户上的标签分布差异

**第三阶段（第3-4周）：全量切换**
- 全量切换到v2.0（锁定标准 + 数据清洗 + k=5分位数方案）
- 建立SPC监控看板
- 编写RFM指标身份证
- 孙工的工作从"每周五跑脚本"变成"每周一看SPC告警"

</details>


### 题2：沙盘变体 — 改变约束条件

**场景**：假设鲜达在6个月后被某巨头收购，用户量从300万暴涨到3000万（接入了巨头的用户池）。同时，鲜达从"纯生鲜电商"扩展为"生鲜+日用+美妆"的综合平台。

**你的RFM系统需要怎么调整？**

<details><summary>点击查看解题思路</summary>

1. **分箱方法**：3000万用户 -> K-Means在3000万样本上计算太慢 -> 用分位数(0-25%/25-50%/50-75%/75-100%)替代，兼顾速度和可解释性
2. **品类差异**：不同品类的购买频率天差地别（生鲜每周买、美妆每月买、日用随机买）-> 不能一刀切用同一个P50 -> **按品类分别计算RFM，再做品类间交叉**
3. **中位数计算**：3000万用户精确中位数计算耗时长 -> 用近似中位数(PERCENTILE_APPROX)或抽样计算
4. **SQL性能**：增加分区(按月份+品类)和索引 -> Spark SQL替代MySQL
5. **SPC控制限**：用户量暴增 -> 群体占比的方差变小 -> 需要重新校准控制限（不能用旧基线）

**最关键的调整**：品类拆分。如果不拆分，生鲜的高频用户(F=20次/月)和日用的低频用户(F=1次/月)混在一起 -> 中位数被拉偏 -> 分群失效。解决方案：分品类计算RFM -> 用户在生鲜品类是"重要价值"，在美妆品类可能是"新用户" -> 运营策略按品类定制。

</details>


### 题3：边界判断 — 什么场景下这些工具不适用

| 场景 | 不适用的工具 | 原因 | 替代方案 |
|------|------------|------|---------|
| 新上线7天的产品，只有800个用户 | K-Means聚类 | 样本太小,聚类不稳定 | 直接用业务规则(如R<3天=活跃) |
| B2B SaaS，客户数只有200家企业 | RFM中位数法 | 200个样本P50没有统计意义 | 用业务阈值(ARR>10万=高价值) |
| 线下零售(无用户ID，只有会员卡) | R值(最近消费距今) | 线下消费不记录user_id | 用会员ID+消费积分替代 |
| 超高频交易产品(外卖，日下单) | F值(90天购买次数) | 90天100次和101次没有本质区别 | F改成"活跃天数"而非"购买次数" |
| 纯内容产品，无交易 | M值(消费金额) | 完全无意义 | 主课2.11已覆盖：M改为互动量 |
| 产品有强季节性(月饼/年货) | 全年统一P50标准 | 中秋节前后的购买行为完全不同 | 按季节/节日分别计算RFM |

**关键认知**：每个工具都有适用边界。知道"什么时候不能用"比知道"怎么用"更能体现分析师的成熟度。


### 题4：落地 — 从RFM系统到运营决策看板

**任务**：设计一个"RFM运营决策看板"（赵总每周一早上打开的第一件事）。要求涵盖：群体分布 + 迁移预警 + 策略推荐 + 数据质量。

<details><summary>点击查看看板设计方案</summary>

| 模块 | 指标 | 可视化 | 说明 |
|------|------|--------|------|
| 群体概览 | 8群体用户数+占比(本周 vs 上周 vs 上月) | 堆叠柱状图 | 一眼看群体结构变化 |
| 迁移预警 | 关键迁移路径(重要价值->重要挽留, 重要发展->重要价值) | 桑基图/热力图 | 重点标注高危迁移 |
| SPC监控 | 8群体占比的控制图(24周滚动) | 折线图+控制限 | 超出控制限自动标红 |
| 策略推荐 | 按群体输出本周运营建议 | 表格(可导出) | 自动匹配最佳券额+触达方式 |
| 数据质量 | 退货率/拆单率/囤货率 周趋势 | 仪表盘 | 污染率>10%告警 |
| 覆盖漏斗 | 运营策略覆盖率(发了券的用户/应发券的用户) | 漏斗图 | 防止"策略只在PPT里" |

**技术架构**：MySQL(数据层) -> Airflow每周一跑SQL流水线 -> 写结果到RFM结果表 -> Metabase/Superset(可视化) -> 周一自动推送到企微群

</details>


---

# 第五部分：知识网络图（10分钟）

## 概念关系全景表

| # | 概念 | 深度 | 重要度 | 关系 | 与核心概念(RFM)的关联 |
|---|------|------|--------|------|---------------------|
| 1 | K-Means自动化分箱 | [M][C] | 必修 | 前置 | 在"切8份"之前——中位数真的是最好的切法吗？ |
| 2 | 轮廓系数+肘部法则 | [M][C] | 必修 | 并行 | 如果K-Means，k应该是几？ |
| 3 | RFM标签边界敏感性 | [M][C] | 必修 | 防护 | 标签贴完后——有多稳定？翻转了多少？ |
| 4 | RFM数据质量三重污染 | [M][C] | 必修 | 防护 | 数据脏了——分群结果还有意义吗？ |
| 5 | RFM完整SQL生产实现 | [C] | 必修 | 落地 | 从Python脚本到生产级SQL流水线 |
| 6 | 客户健康度评分 | [B][C] | 选修 | 替代 | 当"只看交易"不够时——补上行为+反馈 |
| 7 | RFM产品化+SPC监控 | [M][C] | 选修 | 落地 | 从"一次性分析"到"持续监控系统" |

## 概念协同关系图

```
         [1] K-Means自动化分箱
          |
          +--> [2] 轮廓系数+肘部法则
          |
          v
    RFM核心(主课) ---- [3] 边界敏感性(防护)
          |               |
          |               +--> 锁定标准 + 软分群
          |
          +---- [4] 数据质量(防护)
          |         |
          |         +--> 退货/拆单/囤货检测
          |
          +---- [5] SQL生产实现(落地)
          |         |
          |         +--> 5步流水线 + 迁移矩阵
          |
          +---- [6] Health Score(替代)
          |         |
          |         +--> 行为+反馈 扩充RFM
          |
          +---- [7] SPC监控(落地)
                    |
                    +--> 控制图 + 指标身份证
```

## 三版本覆盖对比

| 知识域 | 主课程W2D5 | 本拓展新增 | 状态 |
|--------|-----------|----------|------|
| RFM三值计算+中位数分界 | 已覆盖+完整代码 | — | 完成 |
| 8群体标签+运营策略 | 已覆盖 | — | 完成 |
| 迁移矩阵 | 已覆盖+完整代码 | — | 完成 |
| 非交易场景改写 | 已覆盖+完整代码 | — | 完成 |
| K-Means自动分箱 | — | 完整代码+对比分析 | **新增** |
| 聚类质量评估 | — | 轮廓系数+肘部法则 | **新增** |
| 标签边界敏感性量化 | — | 翻转率分解+噪声检测 | **新增** |
| 数据质量三重污染 | — | 退货/拆单/囤货检测器 | **新增** |
| RFM完整SQL实现 | — | 5步流水线(含迁移矩阵) | **新增** |
| 客户健康度评分 | — | 行为+反馈扩充RFM | **新增** |
| SPC统计过程监控 | — | 控制图+指标身份证 | **新增** |

## 跨模块连接

| 方向 | 关联内容 | 关系 |
|------|---------|------|
| <- W2D4 留存LTV | Bootstrap CI + 生存分析 | W2D4的Bootstrap可用于评估RFM群体占比的置信区间 |
| <- W2D3 漏斗分析 | 转化漏斗 | RFM群体间的迁移路径本质上是一个漏斗 |
| -> W3D1 用户画像 | 多维分群 | Health Score(概念6)是RFM+画像的桥梁 |
| -> W5 SQL | 窗口函数+CTE | 概念5的SQL实现用到ROW_NUMBER/窗口函数 |
| -> W7-W8 AB实验 | 策略验证 | 概念7的SPC可检测AB实验是否改变了用户群体分布 |
| -> W11 ML | K-Means深入 | 概念1-2的K-Means在W11会深入到更多聚类算法 |

## 延伸阅读

| 主题 | 资源 |
|------|------|
| K-Means聚类 | Scikit-learn文档: Clustering章节 |
| 聚类评估 | "How Many Clusters?" (Pham, Dimov, Nguyen 2005) — 肘部法则的数学基础 |
| 数据质量 | "Data Quality: The Accuracy Dimension" (Jack Olson) |
| SPC | "Understanding Statistical Process Control" (Wheeler & Chambers) |
| Health Score | "Customer Success" (Mehta, Steinman, Murphy) Ch8 |
| SQL中位数 | "SQL Cookbook" (Molinaro) Ch9 — 中位数计算的多种SQL写法 |


---

# 第六部分：元认知自评（15分钟）

## 学习循环自评表

逐条检查。诚实回答——自诊的目的是定位薄弱点，不是打分。

| 检查项 | 是 | 否 | 不确定 |
|--------|----|----|--------|
| 我能解释K-Means和中位数法在RFM分箱上的本质区别 | | | |
| 我能在给定数据集上用轮廓系数+肘部法则选择最优k | | | |
| 我能量化RFM标签的噪声翻转率并提出改进方案 | | | |
| 我能用代码检测退货/拆单/囤货对RFM数据的污染 | | | |
| 我能写出RFM的完整SQL流水线(含迁移矩阵) | | | |
| 我能设计一个Health Score来补充RFM的盲区 | | | |
| 我能建立SPC控制图来监控RFM群体的长期趋势 | | | |
| 我知道在什么场景下以上7个工具不适用 | | | |

## 多维度自评量表

每个维度0-2分自评（0=完全不会, 1=需要参考资料, 2=可独立完成）：

| 维度 | 0分 | 2分 | 自评 |
|------|-----|-----|------|
| **概念理解** | 说不出K-Means和轮廓系数是什么 | 能向运营团队解释"为什么k=5比k=8更好" | /2 |
| **代码实现** | 无法写分析代码 | 能独立完成K-Means/污染检测/SPC控制图 | /2 |
| **数学推导** | 不理解轮廓系数的公式 | 能手动计算小数据集的轮廓系数 | /2 |
| **工程落地** | 不知道SQL怎么写RFM | 能写出完整的5步SQL流水线 | /2 |
| **业务判断** | 不知道该不该用Health Score | 能判断何时用RFM、何时加Health Score | /2 |
| **边界判断** | 不知道SPC何时失效 | 能系统性判断各工具的适用条件 | /2 |

**总分：/12**
- 10-12分 -> 拓展掌握扎实，可进入W2D6
- 7-9分 -> 回头重做得分低的维度对应练习
- 4-6分 -> 仔细重读概念网络 + 补充阅读延伸资源
- 0-3分 -> 建议回主课重学W2D5

## 薄弱点精准强化

| 薄弱点 | 强化方案 |
|--------|---------|
| K-Means分箱 | 回概念1代码cell，把k=8改成k=5/10，观察群中心变化 |
| 聚类评估 | 回概念2，修改k_range到2..20，观察轮廓系数和肘部的变化 |
| 边界敏感性 | 回概念3，修改边界宽度(5%/15%/25%)，观察边界用户占比变化 |
| 数据清洗 | 回概念4，分别关闭三个检测器，观察每种污染的独立影响 |
| SQL实现 | 把概念5的SQL复制到你的MySQL环境，用真实数据跑一遍 |
| Health Score | 选一个你熟悉的产品，设计它的Health Score权重分配 |
| SPC监控 | 回概念7，修改控制限宽度(2sigma/2.5sigma/3.5sigma)，观察告警频率变化 |

## 沙盘复述测试

不看上面内容，用自己的话回答：

1. **鲜达的RFM系统最核心的三个问题是什么？**（不超过3句话）
2. **你给赵总的升级方案是什么？分几个阶段？**
3. **你用了哪些工具来支持你的方案？**（至少5个概念名+用途）
4. **如果市场VP反驳"中位数法用了这么多年，为什么要换？"——你怎么回应？**

<details><summary>点击查看参考答案</summary>

1. 中位数一刀切导致15-25%边界用户标签不稳定；大促后数据被退货/拆单/囤货三重污染；Python脚本在300万用户下不可持续、需要SQL自动化。

2. 三个阶段：第一周诊断（K-Means+边界+清洗量化问题）->第二周灰度测试（对10%用户运行新系统）->第三-四周全量切换（锁定标准+清洗+SPC监控）。

3. K-Means（发现最优k≠8）+ 轮廓系数（用数据证明k=5比k=8好）+ 边界敏感性（量化噪声翻转率约30%）+ 数据清洗检测器（证明大促后虚增40%"重要价值"）+ SQL流水线（让孙工从手工操作中解放）。

4. "中位数法本身没错，但有两个问题我们之前忽视了：(1) 每月换中位数标准导致30%的标签翻转是'噪声'(锁定标准可以消除)；(2) 大促后的数据被污染了我们没检测(清洗后'重要价值'回归正常)。中位数法升级后依然是核心——只是加了保护和监控。"

</details>

## 循环毕业检查清单

进入W2D6(PDS)之前确认：
- [ ] 我能向运营团队解释中位数分箱的边界问题并提出改进
- [ ] 我能用K-Means探索用户分群结构并用轮廓系数评估质量
- [ ] 我在每期RFM更新前会自动检查数据质量（退货/拆单/囤货）
- [ ] 我能在SQL中写出完整的RFM流水线（不只是Python）
- [ ] 我了解Health Score应该在什么时候补充RFM
- [ ] 我能建立SPC控制图让RFM从"分析工具"变成"监控系统"
- [ ] 我知道每个工具的适用边界和失效条件

**全部打勾 = W2D5拓展毕业**

---

*03analyst_thinking · Week 2 Day 5 拓展 · 2026-07-01*
*本拓展填补 3 个永久空白区概念 + 4 个核心深化/配套概念*
*下一站：W2D6 PDS日活拆解——用RFM的群体视角来理解DAU的质量构成*
